<a href="https://colab.research.google.com/github/annapozdn/apozdn/blob/main/Qwen3_TTS_Voice_Cloning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import soundfile as sf
import gradio as gr
import numpy as np
from qwen_tts import Qwen3TTSModel

print("Loading model on CPU...")
model = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-1.7B-Base",
    device_map="cpu",
    dtype=torch.float32,
    attn_implementation="eager",
)
print("Model loaded!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 7.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 54.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for flash-attn
  Running setup.py clean for flash-attn
Failed to build flash-attn
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (flash-attn)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.4/61.4 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 4.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.5/113.5 kB 12.9 MB/s et

In [2]:
import torch
import soundfile as sf
import gradio as gr
import numpy as np
from qwen_tts import Qwen3TTSModel

print("Loading Qwen3-TTS Base model...")

model = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-1.7B-Base",
    device_map="cuda:0",
    dtype=torch.bfloat16,
    attn_implementation="eager",
)

print("Model loaded!")


********
********
 
Loading Qwen3-TTS Base model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/245 [00:00<?, ?B/s]

...

preprocessor_config.json:   0%|          | 0.00/127 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

Model loaded!


In [3]:
def clone_voice(new_text, language, reference_audio, ref_transcript):
    """
    Clone a voice and generate speech in that cloned voice.

    Args:
        new_text: The text you want the cloned voice to speak
        language: Target language for synthesis
        reference_audio: Audio file path (3-10 seconds of clear speech)
        ref_transcript: Exact transcript of what's said in reference audio
    """
    try:
        if reference_audio is None:
            return None, "❌ Please upload a reference audio file (3-10 seconds recommended)"

        if not ref_transcript or ref_transcript.strip() == "":
            return None, "❌ Please provide the transcript of your reference audio"

        # Generate voice clone using official Qwen3-TTS API
        wavs, sr = model.generate_voice_clone(
            text=new_text,
            language=language,
            ref_audio=reference_audio,
            ref_text=ref_transcript,
        )

        # Handle output format
        if isinstance(wavs, (list, tuple)):
            audio_data = np.array(wavs[0])
        else:
            audio_data = np.array(wavs)

        return (int(sr), audio_data), f"✅ Voice cloned successfully! Sample rate: {sr}Hz"

    except Exception as e:
        import traceback
        return None, f"❌ Error: {str(e)}\n\n{traceback.format_exc()}"

# Supported languages (10 major languages)
languages = [
    "Chinese", "English", "Japanese", "Korean",
    "German", "French", "Russian", "Portuguese",
    "Spanish", "Italian"
]

# Create Gradio interface
interface = gr.Interface(
    fn=clone_voice,
    inputs=[
        gr.Textbox(
            label="📝 NEW Text (what you want the cloned voice to say)",
            placeholder="Enter the text you want to speak in the cloned voice...",
            lines=4,
            value="Hello! This is my cloned voice speaking new words."
        ),
        gr.Dropdown(
            choices=languages,
            value="English",
            label="🌐 Language"
        ),
        gr.Audio(
            label="🎤 Reference Audio (3-10 seconds of clear speech)",
            type="filepath",
            sources=["upload", "microphone"]
        ),
        gr.Textbox(
            label="📄 Reference Audio Transcript",
            placeholder="Type EXACTLY what is spoken in the reference audio above...",
            lines=3,
            value=""
        )
    ],
    outputs=[
        gr.Audio(label="🔊 Cloned Voice Output", type="numpy"),
        gr.Textbox(label="📊 Status", lines=2)
    ],
    title="🎙️ Qwen3-TTS Voice Cloning",
    description="""
    **Clone any voice from just 3 seconds of audio!**

    **How to use:**
    1. Upload a 3-10 second audio clip of the voice you want to clone
    2. Type exactly what is said in that audio (the transcript)
    3. Enter the new text you want this voice to speak
    4. Click Submit and wait for your cloned voice!

    **Tips for best results:**
    - Use clear audio without background noise
    - Make sure your transcript matches the audio exactly
    - Longer reference audio (10-20 seconds) gives better results
    - Works in 10 languages!
    """,
    examples=[
        [
            "Welcome to my AI voice cloning demo!",
            "English",
            None,
            ""
        ],
        [
            "This technology is amazing!",
            "English",
            None,
            ""
        ]
    ]
)

# Launch the interface
print("🚀 Launching Gradio interface...")
interface.launch(share=True, debug=False)


🚀 Launching Gradio interface...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://bf181f2808885bc4f1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
